# Review Classifier — Remote Processing Server
Run this notebook in Colab. It loads your ML models once and exposes them via a FastAPI server tunnelled through ngrok.

**Steps:**
1. Run Cell 1 to install dependencies
2. Run Cell 2 to set your ngrok token
3. Run Cell 3 to start the server
4. Copy the printed public URL into your local `.env` as `COLAB_API_URL`

In [1]:
# Cell 1 — Install dependencies
!pip install fastapi uvicorn pyngrok nest-asyncio transformers sentence-transformers scipy torchvision -q

In [2]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ""

# import torch
# torch.cuda.is_available()  # should print False

In [3]:
# Cell 2 — Set your ngrok auth token
# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
from pyngrok import ngrok

NGROK_TOKEN = "3DJoDG304UA5Mw6gbn2fQgwkhnt_4q7KBYYTnSgg3CRuF7EUN"  # <-- paste your token here
ngrok.set_auth_token(NGROK_TOKEN)

In [4]:
# @title
# Cell 3 — Load models and start server
import nest_asyncio
import uvicorn
import re
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from scipy.cluster.hierarchy import linkage
from pyngrok import ngrok

nest_asyncio.apply()

# ── Load models once at startup ──────────────────────────────────────────────
print("Loading sentiment classifier...")
sentiment_classifier = pipeline(
    "text-classification",
    # model="cardiffnlp/twitter-roberta-base-sentiment-latest", # doesnt work with colab's GPU
    # model="distilbert-base-uncased-finetuned-sst-2-english",
    model="lxyuan/distilbert-base-multilingual-cased-sentiments-student",
    # model="yangheng/deberta-v3-base-absa-v1.1", # CHAUS GPU like a madafaka
    device=0  # GPU; change to -1 if no GPU
)

print("Loading sentence transformer...")
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Models loaded.")

# ── App ───────────────────────────────────────────────────────────────────────
app = FastAPI()

# increase body size limit
from starlette.middleware.base import BaseHTTPMiddleware
from starlette.requests import Request

@app.middleware("http")
async def set_body_size(request: Request, call_next):
    request._body_size_limit = 100 * 1024 * 1024  # 100MB
    return await call_next(request)


# ── /classify ─────────────────────────────────────────────────────────────────
class ClassifyRequest(BaseModel):
    texts: List[str]  # combined title + review strings

class ClassifyResponse(BaseModel):
    predicted: List[str]
    scores: List[float]

@app.post("/classify", response_model=ClassifyResponse)
def classify(req: ClassifyRequest):
    try:
        all_predicted = []
        all_scores = []
        chunk_size = 500  # process 500 at a time

        for i in range(0, len(req.texts), chunk_size):
            chunk = req.texts[i:i + chunk_size]
            results = sentiment_classifier(chunk, batch_size=32, truncation=True)
            sentiment_map = {
                'POSITIVE': 'positive', 'NEGATIVE': 'negative', 'NEUTRAL': 'neutral',
                'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral',
            }
            all_predicted.extend([sentiment_map.get(r['label'], 'neutral') for r in results])
            all_scores.extend([r['score'] for r in results])

        return ClassifyResponse(predicted=all_predicted, scores=all_scores)
    except Exception as e:
        import traceback
        raise HTTPException(status_code=500, detail=traceback.format_exc())


# ── /cluster ──────────────────────────────────────────────────────────────────
class ClusterRequest(BaseModel):
    categories: List[str]  # raw category strings from df['categories'].dropna().unique()

class ClusterResponse(BaseModel):
    category_names: List[str]
    corpus: List[str]
    linkage_matrix: List[List[float]]  # serialised Z matrix

@app.post("/cluster", response_model=ClusterResponse)
def cluster(req: ClusterRequest):
    try:
        rm_punkt = lambda x: re.sub(r'[^\w\s]', '', x)
        cat_stopwords = {'electronics', 'new', 'all', 'frys', 'e', 'computers',
                         'all tablets', 'tablets', 'amazon'}

        corpus = []
        category_names = []

        for category in req.categories:
            terms = set()
            for term in str(category).split(','):
                cleaned = rm_punkt(term.lower().strip())
                if cleaned and cleaned not in cat_stopwords:
                    terms.add(cleaned)
            corpus.append(' '.join(terms))
            category_names.append(category)

        embeddings = sentence_model.encode(corpus)
        Z = linkage(embeddings, method='ward')

        return ClusterResponse(
            category_names=category_names,
            corpus=corpus,
            linkage_matrix=Z.tolist()
        )
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


# ── Start tunnel + server ────────────────────────────────────────────────────
public_url = ngrok.connect(8000)
print("\n" + "="*60)
print(f"  COLAB_API_URL = {public_url}")
print("  Copy this into your local .env file")
print("="*60 + "\n")

import threading

def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Keep the cell alive
import time
while True:
    time.sleep(60)

Loading sentiment classifier...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loading sentence transformer...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

INFO:     Started server process [4555]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



  COLAB_API_URL = NgrokTunnel: "https://baritone-extending-rake.ngrok-free.dev" -> "http://localhost:8000"
  Copy this into your local .env file



You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


INFO:     2003:e1:3738:701b:afeb:e97d:d0b5:434c:0 - "POST /classify HTTP/1.1" 200 OK
INFO:     2003:e1:3738:701b:afeb:e97d:d0b5:434c:0 - "POST /cluster HTTP/1.1" 200 OK


KeyboardInterrupt: 